<a href="https://colab.research.google.com/github/yuhui-0611/ESAA/blob/main/ESAA_OB_WEEK05_1_Code_Review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Calorie Expenditure Prediction**

[link text](https://www.kaggle.com/code/handeharmankaya/calorie-expenditure-prediction#Modelling)

## 코드 흐름 요약

1.  데이터 불러오기
2.  EDA(탐색적 데이터 분석)
3.  타깃 변수 및 관계 시각화
    - 칼로리 분포
    - 성별에 따른 칼로리 분포
    - 상관관계 히트맵
    - 운동 시간과 칼로리 관계
4.  범주형 변수 인코딩
    - BMI: 체중과 키를 종합한 체형 정보
    - BMR: 기초대사량 느낌의 파생 변수
    - Effort: 운동시간 × 심박수 → 운동 강도 비슷한 정보
    - Temp_Diff: 정상 체온 37도와의 차이
    - Intensity: 심박수 × 체온 → 강도 관련 상호작용
    - Temp_per_Minute: 시간 대비 체온
5.  피처 엔지니어링
6.  입력 변수 / 타깃 분리
7.  1차 모델 비교
    - LinearRegression
    - Ridge
    - Lasso
    - ElasticNet
    - SGDRegressor
    - ExtraTreeRegressor
    - GradientBoostingRegressor
    - AdaBoostRegressor
    - XGBRegressor
    > 어떤 모델이 대략 잘 맞는지 1차 탐색
8.  2차 모델 비교
9.  Optuna로 XGBoost 하이퍼파라미터 튜닝
10.  최종 모델 학습
    - 앙상블 추가




---



## 새롭게 알게 된 내용

- 처음에는 여러 모델을 넓게 비교하고, 이후 강한 모델을 좁혀서 튜닝하는 흐름을 취함
- 처음부터 하나의 모델에 몰입하지 않고, 선형모델, 트리 모델, 부스팅 모델을 전반적으로 비교한 뒤, 좋은 성능을 보이는 XGBoost 계열로 집중함
- 즉, 실전에서는 무조건 처음부터 튜닝하는 것보다, 먼저 후보 모델을 넓게 비교하는 과정이 중요함을 알 수 있음
- algo_test(), smart_algo_test()처럼 모델 비교를 함수로 만들어두면 반복 실험이 훨씬 쉬워진다는 것을 알게 됨
- 즉, 한 번 쓰고 버리는 코드보다 재사용 가능한 구조의 코드가 좋다는 점을 배울 수 있었음



---




< 1차 모델 함수 >
- **스케일링, train/test 분리, 여러 모델 학습, RMSE, MAE, R² 비교**를 한 번에 수행하는 함수
```
def algo_test(x,y):
        L=LinearRegression()
        R=Ridge()
        Lass=Lasso()
        E=ElasticNet()
        sgd=SGDRegressor()
        ETR=ExtraTreeRegressor()
        GBR=GradientBoostingRegressor()
        ada=AdaBoostRegressor()
        xgb=XGBRegressor()
        
        algos=[L,R,Lass,E,sgd,ETR,GBR,ada,xgb]
        algo_names=['Linear','Ridge','Lasso','ElasticNet','SGD','Extra Tree','Gradient Boosting',
                    'AdaBoost','XGBRegressor']
    
        x=MinMaxScaler().fit_transform(x)
                
        x_train, x_test, y_train, y_test=train_test_split(x,y,test_size=.20,random_state=42)
        
        r_squared= []
        rmse= []
        mae= []

        result=pd.DataFrame(columns=['R_Squared','RMSE','MAE'],index=algo_names)

        for algo, name in zip(algos, algo_names):
            p=algo.fit(x_train,y_train).predict(x_test)
            r_squared.append(r2_score(y_test,p))
            rmse.append(mean_squared_error(y_test,p)**.5)
            mae.append(mean_absolute_error(y_test,p))
        
        result.R_Squared=r_squared
        result.RMSE=rmse
        result.MAE=mae

        rtable=result.sort_values('R_Squared',ascending=False)

        return rtable
```


< Optuna 하이퍼파라미터 튜닝 >
```
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 1000, 3000),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'tree_method': 'hist',
        'device': 'cuda',       
        'random_state': 42,
        'n_jobs': -1}
    
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train,eval_set=[(X_test, y_test)],verbose=False)
    preds = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    return rmse

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=20)
```

< 앙상블 추가 >
```
ensemble_preds = (xgb_preds * 0.5) + (cat_preds * 0.5)
```

